# HW2 — Data Cleaning and Validation
Jefferson Li

Inspect `ohlc_hw`, record quality issues, and save validated observations to `ohlc_hw_cleaned` in the local working database.

## 1.1 Load and inspect

| Column | Meaning |
|---|---|
| `ts` | Observation date |
| `ticker` | Raw stock identifier; formatting may be inconsistent |
| `open`, `high`, `low`, `close` | Daily opening, highest, lowest, and closing prices |
| `adj_close` | Vendor-adjusted close; raw high/low bounds do not apply |
| `volume` | Reported trading volume |

In [1]:
from pathlib import Path
from html import unescape
import sqlite3

import numpy as np
import pandas as pd

DB_PATH = Path("data/data.db")
assert DB_PATH.is_file(), "Launch Jupyter from this homework folder."
PRICE_COLS = ["open", "high", "low", "close", "adj_close"]
RAW_PRICES = PRICE_COLS[:4]
SOURCE_COLS = ["ts", "open", "high", "low", "close", "adj_close", "volume", "ticker"]


def normalize_tickers(labels: pd.Series) -> pd.Series:
    """Remove HTML formatting and standardize labels, preserving missing values."""
    return (
        labels.astype("string")
        .str.replace(r"<[^>]*>", "", regex=True)
        .map(lambda value: unescape(value) if pd.notna(value) else pd.NA)
        .astype("string")
        .str.strip()
        .str.upper()
    )


def prepare_data(frame):
    """Normalize identifiers on a copy, preserving original ticker labels."""
    data = frame.copy()
    data["ticker_raw"] = data.get("ticker_raw", data["ticker"])
    data["ticker"] = normalize_tickers(data["ticker"])
    data["ts"] = pd.to_datetime(data["ts"], errors="coerce").dt.normalize()
    return data

In [2]:
conn = sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)
try:
    raw_df = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    reference_df = pd.read_sql_query("SELECT * FROM ohlc", conn)
finally:
    conn.close()

diagnostic_df = prepare_data(raw_df)
normalization_audit = (
    diagnostic_df.groupby(["ticker_raw", "ticker"], dropna=False)
    .size().rename("rows").reset_index()
)
print("Source rows and columns:", raw_df.shape)
display(raw_df.head(3))

Source rows and columns: (397711, 8)


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.43,5.48,5.36,5.44,1.64000,446392,CAT
1,1978-04-05,4.84,4.89,4.81,4.88,0.65699,355200,CVX
2,2018-05-25,124.54,124.76,121.22,122.19,92.22000,8936649,CVX


In [3]:
raw_df.info()
display(raw_df.describe())
parsed_dates = pd.to_datetime(raw_df["ts"], errors="coerce")
print("Date range:", parsed_dates.min().date(), "to", parsed_dates.max().date())
print("Nonmissing dates that failed parsing:", int((raw_df["ts"].notna() & parsed_dates.isna()).sum()))

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Date range: 1962-01-02 to 2024-12-06
Nonmissing dates that failed parsing: 0


In [4]:
print("Ticker labels:", raw_df["ticker"].nunique(), "raw labels ->", diagnostic_df["ticker"].nunique(), "normalized tickers")
print("Tickers:", sorted(diagnostic_df["ticker"].dropna().unique()))
print("Examples of labels needing normalization:")
display(normalization_audit.loc[normalization_audit["ticker_raw"].ne(normalization_audit["ticker"])].head(3))
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percent": raw_df.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)
display(missing_summary)

Ticker labels: 150 raw labels -> 30 normalized tickers
Tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']
Examples of labels needing normalization:


,ticker_raw,ticker,rows
0,"<a target=""_blank""><span style=""color: blue"">A...",AAPL,307
1,"<a target=""_blank""><span style=""color: blue"">A...",AMGN,294
2,"<a target=""_blank""><span style=""color: blue"">A...",AMZN,185


,missing_count,missing_percent
adj_close,19855,4.992319
ts,0,0.000000
open,0,0.000000
high,0,0.000000
low,0,0.000000
close,0,0.000000
volume,0,0.000000
ticker,0,0.000000


## 1.2 Detect data quality issues

The functions below check prices/volume, ticker-specific date gaps, simple raw-close return outliers, and the combined quality report. Returns use unique stock/date observations and sample standard deviation (`ddof=1`); flags are for review. Missing dates are compared with each stock's supplied `ohlc` history.

Severity: **high** for unusable prices/keys or adjustment inconsistencies; **medium** for missing dates, duplicate records, unknown volume, or return-review flags; **low** for label formatting or observed zero volume. An observation can have several issues.

In [5]:
def flag_price_and_volume_issues(frame: pd.DataFrame) -> pd.DataFrame:
    """Return row-level diagnostic masks without modifying the input data."""
    raw_prices = ["open", "high", "low", "close"]
    prices = raw_prices + ["adj_close"]
    flags = pd.DataFrame(index=frame.index)
    for column in prices:
        values = frame[column]
        flags[f"missing_{column}"] = values.isna()
        flags[f"invalid_{column}"] = values.notna() & (
            ~np.isfinite(values) | values.le(0)
        )

    finite_ohlc = np.isfinite(frame[raw_prices]).all(axis=1)
    flags["open_outside_low_high"] = finite_ohlc & (
        frame["open"].lt(frame["low"]) | frame["open"].gt(frame["high"])
    )
    flags["close_outside_low_high"] = finite_ohlc & (
        frame["close"].lt(frame["low"]) | frame["close"].gt(frame["high"])
    )
    flags["high_below_low"] = finite_ohlc & frame["high"].lt(frame["low"])
    volume = frame["volume"]
    flags["missing_volume"] = volume.isna()
    flags["invalid_volume"] = volume.notna() & (
        ~np.isfinite(volume) | volume.lt(0)
    )
    flags["zero_volume"] = volume.eq(0)
    return flags

In [6]:
def find_missing_trading_dates(
    observed: pd.DataFrame, reference: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare ticker-specific date sets; return coverage counts and missing dates.

    Reference dates define each ticker's expected supplied history. Refuse
    unparseable/missing identifiers or observed tickers without a reference.
    """
    actual = observed[["ticker", "ts"]].copy()
    expected = reference[["ticker", "ts"]].copy()
    for frame in (actual, expected):
        frame["ticker"] = normalize_tickers(frame["ticker"])
        frame["ts"] = pd.to_datetime(frame["ts"], errors="raise").dt.normalize()
        if frame[["ticker", "ts"]].isna().any().any() or frame["ticker"].eq("").any():
            raise ValueError("Resolve missing ticker/date identifiers before checking gaps.")
    if expected.empty:
        raise ValueError("The reference calendar is empty.")
    unknown = set(actual["ticker"]) - set(expected["ticker"])
    if unknown:
        raise ValueError(f"No reference dates for observed tickers: {sorted(unknown)}")

    actual_sets = actual.groupby("ticker")["ts"].agg(set).to_dict()
    coverage_rows, missing_rows = [], []
    for ticker, dates in expected.groupby("ticker")["ts"]:
        expected_dates = set(dates)
        actual_dates = actual_sets.get(ticker, set())
        missing = sorted(expected_dates - actual_dates)
        coverage_rows.append({
            "ticker": ticker,
            "reference_start": min(expected_dates),
            "reference_end": max(expected_dates),
            "expected_dates": len(expected_dates),
            "observed_reference_dates": len(expected_dates & actual_dates),
            "missing_dates": len(missing),
            "dates_outside_reference": len(actual_dates - expected_dates),
        })
        missing_rows.extend({"ticker": ticker, "issue_date": date} for date in missing)
    return (
        pd.DataFrame(coverage_rows),
        pd.DataFrame(missing_rows, columns=["ticker", "issue_date"]),
    )

In [7]:
def flag_return_outliers(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return sorted observations with two-tailed 3-sigma flags and a ticker summary.

    Input must already have normalized tickers, parsed dates, and unique keys.
    Reject conflicting/repeated keys rather than choosing a record arbitrarily.
    """
    data = frame.sort_values(["ticker", "ts"]).copy()
    if data[["ticker", "ts"]].isna().any().any():
        raise ValueError("Resolve missing ticker/date identifiers before returns.")
    if data.duplicated(["ticker", "ts"]).any():
        raise ValueError("Returns require unique (ticker, ts) observations.")

    close = pd.to_numeric(data["close"], errors="coerce")
    previous_close = close.groupby(data["ticker"]).shift(1)
    valid_pair = (
        np.isfinite(close) & close.gt(0)
        & np.isfinite(previous_close) & previous_close.gt(0)
    )
    data["previous_close"] = previous_close
    data["simple_return"] = (
        (close.where(valid_pair) - previous_close.where(valid_pair))
        / previous_close.where(valid_pair)
    ).replace([np.inf, -np.inf], np.nan)

    grouped_returns = data.groupby("ticker")["simple_return"]
    data["mean_return"] = grouped_returns.transform("mean")
    data["std_return"] = grouped_returns.transform("std", ddof=1)
    data["screen_available"] = (
        np.isfinite(data["mean_return"])
        & np.isfinite(data["std_return"])
        & data["std_return"].gt(0)
    )
    deviation = data["simple_return"] - data["mean_return"]
    data["return_outlier"] = (
        data["simple_return"].notna() & data["screen_available"]
        & deviation.abs().gt(3 * data["std_return"])
    )
    data["return_z_score"] = deviation / data["std_return"].where(data["screen_available"])
    data["outlier_tail"] = np.select(
        [data["return_outlier"] & deviation.gt(0), data["return_outlier"]],
        ["upper", "lower"], default="",
    )

    summary = data.groupby("ticker").agg(
        observations=("ts", "size"),
        valid_returns=("simple_return", "count"),
        mean_return=("mean_return", "first"),
        std_return=("std_return", "first"),
        flagged_returns=("return_outlier", "sum"),
    )
    summary["screen_status"] = np.select(
        [summary["valid_returns"].lt(2), summary["std_return"].eq(0),
         ~np.isfinite(summary["std_return"]) | ~np.isfinite(summary["mean_return"])],
        ["insufficient_returns", "zero_standard_deviation", "undefined_statistics"],
        default="screen_applied",
    )
    summary["flagged_percent"] = (
        100 * summary["flagged_returns"] / summary["valid_returns"].replace(0, np.nan)
    )
    return data, summary.reset_index()

In [8]:
def reference_values(data, reference):
    """Match the supplied reference by stock/date; never carry factors across dates."""
    ref = prepare_data(reference)
    if ref.duplicated(["ticker", "ts"]).any():
        raise ValueError("Reference stock/date keys must be unique.")
    index = pd.MultiIndex.from_frame(data[["ticker", "ts"]])
    matched = ref.set_index(["ticker", "ts"]).reindex(index)
    matched.index = data.index
    factor = matched["adj_close"] / matched["close"]
    good = (np.isfinite(factor) & factor.gt(0)
            & matched["close"].gt(0) & matched["adj_close"].gt(0))
    return matched, factor.where(good)


def make_quality_report(frame, reference, source_issues=True):
    """Return one row per finding, allowing multiple findings per observation."""
    data = prepare_data(frame)
    rows = []

    def add(mask, issue, description, severity):
        found = data.loc[mask.fillna(False), ["ticker", "ts"]].copy()
        found = found.rename(columns={"ts": "issue_date"})
        found["issue_type"] = issue
        found["description"] = description
        found["severity"] = severity
        rows.append(found)

    for issue, mask in flag_price_and_volume_issues(data).items():
        severity = "low" if issue == "zero_volume" else "high"
        if issue == "missing_volume":
            severity = "medium"
        add(mask, issue, issue.replace("_", " "), severity)
    add(data["ticker"].isna() | data["ticker"].eq("") | data["ts"].isna(),
        "invalid_key", "Missing or invalid stock/date identifier", "high")
    if source_issues:
        add(data["ticker_raw"].ne(data["ticker"]), "ticker_format",
            "HTML/whitespace/case normalized; original label retained", "low")
    add(data.duplicated(SOURCE_COLS), "exact_duplicate",
        "Repeated record beyond its first occurrence", "medium")
    unique = data.drop_duplicates(SOURCE_COLS)
    conflict = unique.duplicated(["ticker", "ts"], keep=False)
    conflict_keys = pd.MultiIndex.from_frame(unique.loc[conflict, ["ticker", "ts"]])
    add(pd.Series(pd.MultiIndex.from_frame(data[["ticker", "ts"]]).isin(conflict_keys), index=data.index),
        "conflicting_key", "Different records share a normalized stock/date key", "high")
    _, factor = reference_values(data, reference)
    expected_adjusted = data["close"] * factor
    mismatch = (factor.notna() & data["adj_close"].notna() & data["close"].gt(0)
                & ~np.isclose(data["adj_close"], expected_adjusted, rtol=1e-7, atol=1e-9))
    add(mismatch, "adjustment_mismatch", "Adjusted close disagrees with date-matched reference factor", "high")

    eligible = unique.loc[~conflict & unique[["ticker", "ts"]].notna().all(axis=1)]
    if not eligible.empty:
        returns, _ = flag_return_outliers(eligible)
        add(pd.Series(data.index.isin(returns.index[returns["return_outlier"]]), index=data.index),
            "return_outlier", "Simple raw-close return beyond 3 sample standard deviations; review, not delete", "medium")
    _, missing = find_missing_trading_dates(data, reference)
    if not missing.empty:
        missing["issue_type"] = "missing_trading_date"
        missing["description"] = "Absent stock/date row relative to supplied ticker-specific history"
        missing["severity"] = "medium"
        rows.append(missing)
    columns = ["ticker", "issue_type", "issue_date", "description", "severity"]
    return pd.concat(rows, ignore_index=True)[columns].sort_values(["ticker", "issue_date", "issue_type"])

In [9]:
quality_before = make_quality_report(raw_df, reference_df)
print("Quality findings before repairs:")
display(quality_before["issue_type"].value_counts().rename("findings").to_frame())
print("Example report rows, one per issue type:")
display(quality_before.groupby("issue_type", sort=False).head(1))
trading_date_summary, missing_trading_dates = find_missing_trading_dates(diagnostic_df, reference_df)
print("Missing stock/date observations:", len(missing_trading_dates), "across", len(trading_date_summary), "tickers")

Quality findings before repairs:


,findings
issue_type,
ticker_format,39936
exact_duplicate,29460
missing_adj_close,19855
adjustment_mismatch,11265
return_outlier,5056
zero_volume,81
invalid_open,5
open_outside_low_high,5


Example report rows, one per issue type:


,ticker,issue_type,issue_date,description,severity
39764,AAPL,ticker_format,1980-12-19,HTML/whitespace/case normalized; original labe...,low
105617,AAPL,return_outlier,1980-12-26,Simple raw-close return beyond 3 sample standa...,medium
68786,AAPL,exact_duplicate,1981-01-02,Repeated record beyond its first occurrence,medium
12003,AAPL,missing_adj_close,1981-01-08,missing adj close,high
93001,AAPL,adjustment_mismatch,1981-02-03,Adjusted close disagrees with date-matched ref...,high
19907,AXP,zero_volume,1972-12-14,zero volume,low
2,CVX,invalid_open,1962-01-02,invalid open,high
19862,CVX,open_outside_low_high,1962-01-02,open outside low high,high


Missing stock/date observations: 0 across 30 tickers


## 1.3 Clean and validate

Remove exact duplicates and quarantine conflicting keys or unresolved prices. Forward-fill raw prices only within a sorted ticker and recheck bounds. Unknown volume becomes a flagged zero placeholder. Reconstruct adjusted close with a valid same-date reference factor; never carry factors across dates. Interpolate only explicitly confirmed close errors between valid neighbors.

Retain return outliers with review flags and evidence/uncertainty notes. Validate the result before saving, keeping the raw tables unchanged.

In [10]:
def clean_market_data(frame, reference, confirmed_close_errors=()):
    """Repair a working copy, retaining flags and quarantining unresolved rows.

    confirmed_close_errors is an explicitly reviewed set of (ticker, ISO-date)
    keys. Only those close errors are interpolated between valid neighbors.
    """
    data = prepare_data(frame)
    data["source_row"] = np.arange(len(data))
    duplicate = data.duplicated(SOURCE_COLS)
    duplicate_count = int(duplicate.sum())
    data = data.loc[~duplicate].copy()
    all_missing = data[RAW_PRICES].isna().all(axis=1)
    all_missing_count = int(all_missing.sum())
    data = data.loc[~all_missing].copy()
    quarantined = []

    def quarantine(mask, reason):
        nonlocal data
        rejected = data.loc[mask].copy()
        rejected["quarantine_reason"] = reason
        quarantined.append(rejected)
        data = data.loc[~mask].copy()

    quarantine(data["ticker"].isna() | data["ticker"].eq("") | data["ts"].isna(), "invalid_key")
    quarantine(data.duplicated(["ticker", "ts"], keep=False), "conflicting_key")
    data = data.sort_values(["ticker", "ts"])
    original = data[PRICE_COLS + ["volume"]].copy()
    matched, factor = reference_values(data, reference)
    data["ticker_normalized"] = data["ticker_raw"].ne(data["ticker"])
    data["adjustment_mismatch"] = (factor.notna() & data["adj_close"].notna()
        & ~np.isclose(data["adj_close"], data["close"] * factor, rtol=1e-7, atol=1e-9))
    data["zero_volume_observed"] = data["volume"].eq(0)

    # Invalid raw prices become missing; ambiguous bounds are checked again later.
    for column in RAW_PRICES:
        data[column] = data[column].where(np.isfinite(data[column]) & data[column].gt(0))
    inverted = data["high"].lt(data["low"])
    data.loc[inverted, ["high", "low"]] = np.nan
    for column in ["open", "close"]:
        outside = data[column].lt(data["low"]) | data[column].gt(data["high"])
        data.loc[outside, column] = np.nan

    confirmed_keys = {(str(t).upper(), pd.Timestamp(d).normalize()) for t, d in confirmed_close_errors}
    confirmed = pd.Series([key in confirmed_keys for key in zip(data["ticker"], data["ts"])], index=data.index)
    data.loc[confirmed, "close"] = np.nan
    interpolated = data.groupby("ticker")["close"].transform(
        lambda values: values.interpolate(method="linear", limit_area="inside")
    )
    for column in RAW_PRICES:
        filled = data.groupby("ticker")[column].ffill()
        data[column] = filled
    # Confirmed errors use interpolation only: no extrapolation at either end.
    data.loc[confirmed, "close"] = interpolated.loc[confirmed]
    data["close_interpolated"] = confirmed & data["close"].notna()
    for column in RAW_PRICES:
        data[f"{column}_repaired"] = ~(data[column].eq(original[column])
                                                     | (data[column].isna() & original[column].isna()))

    invalid_adjusted = ~np.isfinite(data["adj_close"]) | data["adj_close"].le(0)
    needs_adjustment = invalid_adjusted | data["adjustment_mismatch"] | data["close_repaired"]
    data.loc[needs_adjustment & factor.notna(), "adj_close"] = data["close"] * factor
    data.loc[needs_adjustment & factor.isna(), "adj_close"] = np.nan
    data["adj_close_repaired"] = ~(data["adj_close"].eq(original["adj_close"])
        | (data["adj_close"].isna() & original["adj_close"].isna()))
    data["adjustment_factor"] = data["adj_close"] / data["close"]
    data["adjustment_source"] = np.where(needs_adjustment & factor.notna(), "ohlc_date_factor", "original_vendor")
    bad_volume = ~np.isfinite(data["volume"]) | data["volume"].lt(0)
    data["volume_repaired"] = bad_volume
    data["volume_unknown"] = bad_volume
    data.loc[bad_volume, "volume"] = 0

    valid_prices = np.isfinite(data[PRICE_COLS]).all(axis=1) & data[PRICE_COLS].gt(0).all(axis=1)
    valid_bounds = (data["low"].le(data["open"]) & data["open"].le(data["high"])
                    & data["low"].le(data["close"]) & data["close"].le(data["high"]))
    quarantine(~(valid_prices & valid_bounds), "unresolved_price_or_adjustment_or_bounds")
    retained_returns, _ = flag_return_outliers(data)
    data["outlier_review"] = retained_returns["return_outlier"]
    data["outlier_evidence"] = ""
    # Reference agreement supports keeping the supplied observation, not a news/split claim.
    ref_sorted = prepare_data(reference).sort_values(["ticker", "ts"])
    ref_sorted["reference_previous_close"] = ref_sorted.groupby("ticker")["close"].shift(1)
    ref_pairs = ref_sorted.set_index(["ticker", "ts"])
    lookup = ref_pairs.reindex(pd.MultiIndex.from_frame(data[["ticker", "ts"]]))
    pair_agrees = (np.isclose(data["close"], lookup["close"].to_numpy(), equal_nan=False)
        & np.isclose(retained_returns["previous_close"], lookup["reference_previous_close"].to_numpy(), equal_nan=False))
    data.loc[data["outlier_review"], "outlier_evidence"] = "Cause unverified; retained with review flag"
    data.loc[data["outlier_review"] & pair_agrees, "outlier_evidence"] = "Raw-close pair agrees with supplied reference; cause unverified"
    quarantine_df = pd.concat(quarantined, ignore_index=True)
    accounting = pd.Series({
        "input_rows": len(frame), "removed_exact_duplicates": duplicate_count,
        "removed_all_raw_prices_missing": all_missing_count,
        "quarantined_rows": len(quarantine_df), "retained_rows": len(data),
    }, name="rows")
    assert len(frame) == duplicate_count + all_missing_count + len(quarantine_df) + len(data)
    return data, quarantine_df, accounting

In [11]:
def validate_cleaned(data, reference):
    """Check retained keys, prices, bounds, volumes, and reference adjustment consistency."""
    assert not data.empty, "No rows retained."
    assert not data.duplicated(["ticker", "ts"]).any()
    assert data[["ticker", "ts"]].notna().all().all()
    assert np.isfinite(data[PRICE_COLS]).all().all() and data[PRICE_COLS].gt(0).all().all()
    assert np.isfinite(data["volume"]).all() and data["volume"].ge(0).all()
    assert data["low"].le(data["open"]).all() and data["open"].le(data["high"]).all()
    assert data["low"].le(data["close"]).all() and data["close"].le(data["high"]).all()
    _, factor = reference_values(data, reference)
    known = factor.notna()
    assert np.allclose(data.loc[known, "adj_close"], data.loc[known, "close"] * factor[known], rtol=1e-7, atol=1e-9)

In [12]:
cleaned_df, quarantine_df, row_accounting = clean_market_data(raw_df, reference_df)
validate_cleaned(cleaned_df, reference_df)
quality_after = make_quality_report(cleaned_df, reference_df, source_issues=False)
assert not quality_after["severity"].eq("high").any()

print("Row accounting:")
display(row_accounting.to_frame())
repair_counts = cleaned_df[[f"{column}_repaired" for column in PRICE_COLS + ["volume"]]].sum()
print("Repaired values among retained rows:")
display(repair_counts.loc[repair_counts.gt(0)].to_frame("repaired_values"))
print("Before/after quality findings (multiple findings can refer to one observation):")
display(pd.concat({"before": quality_before["issue_type"].value_counts(),
                   "after": quality_after["issue_type"].value_counts()}, axis=1).fillna(0).astype(int))
print("Before/after price statistics:")
display(pd.DataFrame({
    "before_mean": raw_df[PRICE_COLS].mean(),
    "after_mean": cleaned_df[PRICE_COLS].mean(),
    "before_missing": raw_df[PRICE_COLS].isna().sum(),
    "after_missing": cleaned_df[PRICE_COLS].isna().sum(),
}))
display(quarantine_df[["ticker", "ts", "quarantine_reason"]])
print("The five post-cleaning date gaps are quarantined observations. Return thresholds are recomputed after cleaning.")

Row accounting:


,rows
input_rows,397711
removed_exact_duplicates,29460
removed_all_raw_prices_missing,0
quarantined_rows,5
retained_rows,368246


Repaired values among retained rows:


,repaired_values
adj_close_repaired,28791


Before/after quality findings (multiple findings can refer to one observation):


,before,after
issue_type,,
ticker_format,39936,0
exact_duplicate,29460,0
missing_adj_close,19855,0
adjustment_mismatch,11265,0
return_outlier,5056,5057
zero_volume,81,74
invalid_open,5,0
open_outside_low_high,5,0
missing_trading_date,0,5


Before/after price statistics:


,before_mean,after_mean,before_missing,after_missing
open,46.285479,46.309394,0,0
high,46.747660,46.771824,0,0
low,45.816279,45.840375,0,0
close,46.292903,46.317325,0,0
adj_close,51.652250,37.274670,19855,0


,ticker,ts,quarantine_reason
0,CVX,1962-01-02,unresolved_price_or_adjustment_or_bounds
1,JNJ,1962-01-02,unresolved_price_or_adjustment_or_bounds
2,MCD,1966-07-05,unresolved_price_or_adjustment_or_bounds
3,MRK,1962-01-02,unresolved_price_or_adjustment_or_bounds
4,PG,1962-01-02,unresolved_price_or_adjustment_or_bounds


The five post-cleaning date gaps are quarantined observations. Return thresholds are recomputed after cleaning.


In [13]:
def sqlite_ready(frame):
    """Use consistent ISO dates for SQLite filters and keep numeric metric fractions."""
    output = frame.copy()
    for column in ["ts", "issue_date"]:
        if column in output:
            output[column] = pd.to_datetime(output[column]).dt.strftime("%Y-%m-%d")
    return output


# Local audits are excluded from Git along with the working database.
DATA_DIR = DB_PATH.parent
quality_before.to_csv(DATA_DIR / "quality_before.csv", index=False)
quality_after.to_csv(DATA_DIR / "quality_after.csv", index=False)
quarantine_df.to_csv(DATA_DIR / "quarantine.csv", index=False)
normalization_audit.to_csv(DATA_DIR / "ticker_normalization.csv", index=False)
row_accounting.to_csv(DATA_DIR / "row_accounting.csv")

with sqlite3.connect(DB_PATH) as write_conn:
    sqlite_ready(cleaned_df).to_sql("ohlc_hw_cleaned", write_conn, if_exists="replace", index=False)
    sqlite_ready(quality_before).to_sql("hw2_quality_before", write_conn, if_exists="replace", index=False)
    sqlite_ready(quality_after).to_sql("hw2_quality_after", write_conn, if_exists="replace", index=False)
    sqlite_ready(quarantine_df).to_sql("hw2_quarantine", write_conn, if_exists="replace", index=False)
    # Raw instructor tables must remain byte-for-byte equivalent as data.
    pd.testing.assert_frame_equal(raw_df, pd.read_sql_query("SELECT * FROM ohlc_hw", write_conn))
    pd.testing.assert_frame_equal(reference_df, pd.read_sql_query("SELECT * FROM ohlc", write_conn))
    saved = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", write_conn)
validate_cleaned(prepare_data(saved), reference_df)
print(f"Saved and validated {len(saved):,} rows in ohlc_hw_cleaned. Raw tables unchanged.")
print("Post-cleaning missing dates are the documented quarantined observations, not filled prices.")

Saved and validated 368,246 rows in ohlc_hw_cleaned. Raw tables unchanged.
Post-cleaning missing dates are the documented quarantined observations, not filled prices.


## Small validation examples

Separate synthetic data checks the required cases absent from the source: an omitted date, both return tails, zero/undefined variance, missing prices/volume, invalid bounds, conflicting keys, and repair/quarantine limits. All assertions are retained; examples never enter the course database.

In [14]:
demo_reference_dates = pd.DataFrame({
    "ticker": ["ABC", "ABC", "ABC", "DEF", "DEF"],
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-03", "2024-01-04"],
})

demo_observed_dates = demo_reference_dates.drop(index=1)

demo_date_summary, demo_missing_dates = find_missing_trading_dates(
    demo_observed_dates, demo_reference_dates
)

assert demo_missing_dates.to_dict("records") == [
    {"ticker": "ABC", "issue_date": pd.Timestamp("2024-01-03")}
]

assert demo_date_summary.set_index("ticker").loc["DEF", "missing_dates"] == 0

print('Missing-date example passed.')

Missing-date example passed.


In [15]:
demo_moves = [0.0] * 15 + [0.30] + [0.0] * 15 + [-0.30] + [0.0] * 15

demo_swing = pd.DataFrame({
    "ticker": "SWING",
    "ts": pd.date_range("2024-01-01", periods=len(demo_moves) + 1),
    "close": 100 * np.cumprod(np.r_[1.0, 1 + np.array(demo_moves)]),
})

demo_flat = pd.DataFrame({
    "ticker": "FLAT", "ts": pd.date_range("2024-01-01", periods=4), "close": 100.0,
})

demo_short = pd.DataFrame({"ticker": ["SHORT"], "ts": [pd.Timestamp("2024-01-01")], "close": [100.0]})

demo_invalid = pd.DataFrame({
    "ticker": "INVALID", "ts": pd.date_range("2024-01-01", periods=4),
    "close": [100.0, 0.0, 100.0, 100.0],
})

demo_return_input = pd.concat([demo_swing, demo_flat, demo_short, demo_invalid], ignore_index=True)

demo_return_rows, demo_return_summary = flag_return_outliers(demo_return_input)

demo_flagged = demo_return_rows.loc[demo_return_rows["return_outlier"]]

assert set(demo_flagged["outlier_tail"]) == {"upper", "lower"}

assert len(demo_flagged) == 2

assert np.allclose(sorted(demo_flagged["simple_return"]), [-0.30, 0.30])

assert demo_return_summary.set_index("ticker").loc["FLAT", "screen_status"] == "zero_standard_deviation"

assert demo_return_summary.set_index("ticker").loc["SHORT", "screen_status"] == "insufficient_returns"

assert demo_return_rows.loc[demo_return_rows["ticker"].eq("INVALID"), "simple_return"].iloc[1:3].isna().all()

assert demo_return_rows.groupby("ticker").head(1)["simple_return"].isna().all()

print('Return-tail, variance, history, and invalid-price checks passed.')

Return-tail, variance, history, and invalid-price checks passed.


In [16]:
fixture_reference = pd.DataFrame({
    "ticker": ["ABC"] * 6 + ["LEAD"] + ["CONFLICT"] + ["ERR"] * 3 + ["EDGE"] * 3,
    "ts": list(pd.bdate_range("2024-01-02", periods=6))
          + [pd.Timestamp("2024-01-02")] * 2
          + list(pd.bdate_range("2024-01-02", periods=3)) * 2,
})

for column, value in {"open": 100.0, "high": 110.0, "low": 90.0, "close": 100.0,
                      "adj_close": 50.0, "volume": 100.0}.items():
    fixture_reference[column] = value

fixture = fixture_reference.copy()

fixture.loc[1, "open"] = np.nan

fixture.loc[2, "volume"] = np.nan

fixture.loc[3, RAW_PRICES] = np.nan

fixture.loc[4, "adj_close"] = np.nan

fixture.loc[5, "high"] = 80.0

fixture.loc[6, "open"] = 0.0

fixture.loc[9, ["close", "adj_close"]] = [999.0, 500.0]

fixture.loc[11, ["close", "adj_close"]] = [999.0, 500.0]

conflict_row = fixture.loc[[7]].copy()

conflict_row["close"] = 101.0

unknown_factor = fixture.loc[[0]].copy()

unknown_factor["ticker"] = "NOFACTOR"

unknown_factor["adj_close"] = np.nan

fixture = pd.concat([fixture, fixture.iloc[[0]], conflict_row, unknown_factor], ignore_index=True)

fixture_flags = flag_price_and_volume_issues(fixture)

assert fixture_flags["missing_open"].any() and fixture_flags["high_below_low"].any()

fixture_clean, fixture_quarantine, fixture_counts = clean_market_data(
    fixture, fixture_reference,
    confirmed_close_errors=[("ERR", "2024-01-03"), ("EDGE", "2024-01-02")],
)

validate_cleaned(fixture_clean, fixture_reference)

assert fixture_counts["removed_exact_duplicates"] == 1

assert fixture_counts["removed_all_raw_prices_missing"] == 1

assert not fixture_clean["ticker"].isin(["LEAD", "CONFLICT", "NOFACTOR"]).any()

assert len(fixture_quarantine.loc[fixture_quarantine["ticker"].eq("CONFLICT")]) == 2

assert fixture_quarantine["ticker"].eq("EDGE").sum() == 1

assert fixture_clean.loc[fixture_clean["volume_unknown"], "volume"].eq(0).all()

assert fixture_clean.loc[fixture_clean["close_interpolated"], "close"].eq(100).all()

assert fixture_clean["open_repaired"].any() and fixture_clean["adj_close_repaired"].any()

print('Repair, quarantine, and interpolation checks passed.')

Repair, quarantine, and interpolation checks passed.
